# Country Maps
Generates a styled JPG map for each SIDS country showing the country outline and tile grid.

In [2]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

%matplotlib inline

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar

import contextily as ctx

from ldn.grids import get_gadm
from ldn.utils import ALL_COUNTRIES

from dep_tools.grids import (
    COUNTRIES_AND_CODES as DEP_COUNTRIES_AND_CODES,
)
from ldn.utils import NON_DEP_COUNTRIES

from matplotlib.ticker import FuncFormatter
import pyproj

transformer = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
COUNTRIES_PATH = Path("../ldn/gadm_sids.gpkg")
GRID_PATH      = Path("../ldn/sids_all_tiles.geojson")

OUTPUT_DIR = Path("maps")
for subdir in ("", "countries", "countries/Pacific", "countries/Non-Pacific", "regions"):
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

# I already ran this once so no need to rerun.
# get_gadm(countries=ALL_COUNTRIES, overwrite=True) # Update to all countries.

# Load data
countries = gpd.read_file(COUNTRIES_PATH)
grid      = gpd.read_file(GRID_PATH)

# Reproject to Web Mercator for contextily basemap
WEB_MERCATOR = "EPSG:3857"
countries = countries.to_crs(WEB_MERCATOR)
grid      = grid.to_crs(WEB_MERCATOR)

print(f"Loaded {len(countries)} countries")
print(f"Loaded {len(grid)} grid tiles")
print(f"Country columns: {list(countries.columns)}")

Loaded 60 countries
Loaded 839 grid tiles
Country columns: ['GID_0', 'COUNTRY', 'geometry']


In [4]:
NAME_COL = "COUNTRY"
print(f"Using '{NAME_COL}' as country name column")
print(countries[NAME_COL].unique())

Using 'COUNTRY' as country name column
['Anguilla' 'Antigua and Barbuda' 'Aruba' 'Bahamas' 'Barbados' 'Belize'
 'Bermuda' 'British Virgin Islands' 'Cayman Islands' 'Cuba' 'Curaçao'
 'Dominica' 'Dominican Republic' 'Grenada' 'Guadeloupe' 'Guyana' 'Haiti'
 'Jamaica' 'Martinique' 'Montserrat' 'Puerto Rico' 'Saint Kitts and Nevis'
 'Saint Lucia' 'Saint Vincent and the Grenadines' 'Sint Maarten'
 'Suriname' 'Trinidad and Tobago' 'Turks and Caicos Islands'
 'Virgin Islands, U.S.' 'American Samoa' 'Cook Islands' 'Fiji'
 'French Polynesia' 'Guam' 'Kiribati' 'Marshall Islands' 'Micronesia'
 'Nauru' 'New Caledonia' 'Niue' 'Northern Mariana Islands' 'Palau'
 'Papua New Guinea' 'Samoa' 'Solomon Islands' 'Timor-Leste' 'Tonga'
 'Tuvalu' 'Vanuatu' 'Cabo Verde' 'Comoros' 'Guinea-Bissau' 'Maldives'
 'Mauritius' 'São Tomé and Príncipe' 'Seychelles' 'Singapore'
 'Pitcairn Islands' 'Tokelau' 'Wallis and Futuna']


In [8]:
DEP_COUNTRY_NAMES = set(DEP_COUNTRIES_AND_CODES.keys())
NON_DEP_COUNTRY_NAMES = set(NON_DEP_COUNTRIES.keys())

def get_region_label(country_name: str) -> str:
    if country_name in DEP_COUNTRY_NAMES:
        return "Pacific"
    return "Non-Pacific"


REGION_COLORS = {
    "Pacific": {
        "boundary": "#FF003C",
        "grid":     "#00CFFF",
    },
    "Non-Pacific": {
        "boundary": "#FF6B00",
        "grid":     "#39FF14",
    },
}

colors = {
    "grey": "#e1e1e1",
    "dark_grey": "#888888",
    "black": "#000000",
    "white": "#FFFFFF",
}

MARGIN = 1.0
PADDING = 0.7

def add_north_arrow(ax):
    """Simple north arrow positioned with consistent margin from map edge."""
    size=0.055
    # x = 0.95 # Too left
    # y = 0.88 # Too low
    x = 0.97 # Good
    # y = 0.92 # Too high
    y = 0.9
    ax.annotate(
        "",
        xy=(x, y + size), xytext=(x, y),
        xycoords='axes fraction', textcoords='axes fraction',
        arrowprops=dict(arrowstyle="->", color=colors['grey'], lw=2.5),
        zorder=5,
    )
    ax.text(
        x, y + size + 0.005, "N",
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=11, fontweight="bold", color=colors['white'],
        path_effects=[pe.withStroke(linewidth=2, foreground=colors['black'])],
        zorder=5,
    )

# Shared core
def make_map(
    title_main: str,
    title_sub: str,
    geom: gpd.GeoDataFrame,
    grid_tiles: gpd.GeoDataFrame,
    output_path: Path,
    color_boundary: str,
    color_grid: str,
    boundary_label: str = "Country boundary",
):
    """Generate and save a styled map.  Shared by country and region modes."""

    bounds = geom.total_bounds
    x_range = bounds[2] - bounds[0]
    y_range = bounds[3] - bounds[1]

    min_extent = max(x_range, y_range, 50_000)
    pad  = min_extent * 0.1
    half = (min_extent + 2 * pad) / 2
    cx   = (bounds[0] + bounds[2]) / 2
    cy   = (bounds[1] + bounds[3]) / 2

    xlim = (cx - half, cx + half)
    ylim = (cy - half, cy + half)

    fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Basemap
    ctx.add_basemap(
        ax,
        crs=WEB_MERCATOR,
        source=ctx.providers.Esri.WorldImagery,
        zoom="auto",
        attribution=False,
    )

    # Grid tiles
    if not grid_tiles.empty:
        grid_tiles.plot(
            ax=ax,
            facecolor="none",
            edgecolor=color_grid,
            linewidth=1.8,
            alpha=1,
        )

    # Geometry outline
    geom.plot(
        ax=ax,
        facecolor="none",
        edgecolor=color_boundary,
        linewidth=2.5,
    )

    # Two-line title: main in white, subtitle in grid colour
    # Use ax.text so each line can have its own colour
    ax.text(
        0.5, 1.035,
        title_main,
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=18, fontweight="bold",
        color=colors["black"],
    )
    ax.text(
        0.5, 1.008,
        title_sub,
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=13, fontweight="bold",
        color=color_grid,                        # <── coloured to grid
        path_effects=[pe.withStroke(linewidth=2, foreground=colors["black"])],
    )

    # Scale bar
    scalebar = ScaleBar(
        1, units="m", dimension="si-length",
        location="lower left",
        pad=PADDING, border_pad=MARGIN, sep=3,
        color=colors["white"],
        box_color=colors["black"], box_alpha=0.7,
        font_properties={"size": 11},
    )
    ax.add_artist(scalebar)

    add_north_arrow(ax)

    # Legend
    legend_elements = [
        Line2D([0], [0], color=color_boundary, linewidth=2.5, label=boundary_label),
        Line2D([0], [0], color=color_grid,     linewidth=1.8, label="Analysis grid tiles"),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper left",
        framealpha=0.7, fontsize=11,
        labelcolor=colors["white"],
        facecolor=colors["black"],
        borderpad=PADDING, borderaxespad=MARGIN,
        handlelength=1.5,
    )

    # Tick formatters
    def format_lon(x, _):
        lon, _ = transformer.transform(x, 0)
        return f"{lon:.2f}°E" if lon >= 0 else f"{abs(lon):.2f}°W"

    def format_lat(y, _):
        _, lat = transformer.transform(0, y)
        return f"{lat:.2f}°N" if lat >= 0 else f"{abs(lat):.2f}°S"

    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    ax.tick_params(labelsize=10, color=colors["black"])
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_edgecolor(colors["black"])

    ax.text(
        1 - 0.01, 0.01,
        "Basemap © Esri World Imagery | Boundaries © GADM",
        transform=ax.transAxes,
        fontsize=7, color=colors["grey"], va="bottom", ha="right",
    )

    plt.tight_layout()
    fig.savefig(output_path, dpi=150, format="jpg", bbox_inches="tight", pil_kwargs={"quality": 92})
    plt.close(fig)
    print(f"  Saved to {output_path}")


# Convenience wrappers
def make_country_map(country_name: str, country_geom: gpd.GeoDataFrame,
                     grid_tiles: gpd.GeoDataFrame, output_path: Path):
    """Country mode – colours chosen from the country's region."""
    region = get_region_label(country_name)
    c = REGION_COLORS[region]
    make_map(
        title_main=country_name,
        title_sub=region,
        geom=country_geom,
        grid_tiles=grid_tiles,
        output_path=output_path,
        color_boundary=c["boundary"],
        color_grid=c["grid"],
        boundary_label="Country boundary",
    )


def make_region_map(region_name: str, region_geom: gpd.GeoDataFrame,
                    grid_tiles: gpd.GeoDataFrame, output_path: Path):
    """Region mode – dissolves all countries into one outline."""
    c = REGION_COLORS[region_name]
    n_countries = region_geom.shape[0]
    make_map(
        title_main=region_name,
        title_sub=f"{n_countries} countries / territories",
        geom=region_geom,
        grid_tiles=grid_tiles,
        output_path=output_path,
        color_boundary=c["boundary"],
        color_grid=c["grid"],
        boundary_label="Region boundary",
    )

In [10]:
# Generate country maps
country_names = sorted(countries[NAME_COL].unique())
print(f"Generating maps for {len(country_names)} countries...")

for name in country_names:
    print(f"  Processing: {name}")
    subset       = countries[countries[NAME_COL] == name]
    country_grid = grid[grid.intersects(subset.union_all())]
    safe_name    = name.replace(" ", "_").replace("/", "-")
    out_path     = OUTPUT_DIR / "countries" / get_region_label(name) / f"{safe_name}.jpg"
    try:
        make_country_map(name, subset, country_grid, out_path)
    except Exception as e:
        print(f"  Failed for {name}: {e}")

# Generate region maps
REGION_FILTER = {
    "Pacific":     DEP_COUNTRY_NAMES,
    "Non-Pacific": NON_DEP_COUNTRY_NAMES,
}

print("\nGenerating region maps...")
for region_name, country_set in REGION_FILTER.items():
    print(f"  Processing: {region_name}")
    region_geom = countries[countries[NAME_COL].isin(country_set)]
    region_grid = grid[grid.intersects(region_geom.union_all())]
    out_path    = OUTPUT_DIR / "regions" / f"{region_name}.jpg"
    try:
        make_region_map(region_name, region_geom, region_grid, out_path)
    except Exception as e:
        print(f"  Failed for {region_name}: {e}")

print(f"\nDone. Region maps saved to '{OUTPUT_DIR / 'regions'}/'")

Generating maps for 60 countries...
  Processing: American Samoa
  Saved to maps/countries/Pacific/American_Samoa.jpg
  Processing: Anguilla
  Saved to maps/countries/Non-Pacific/Anguilla.jpg
  Processing: Antigua and Barbuda
  Saved to maps/countries/Non-Pacific/Antigua_and_Barbuda.jpg
  Processing: Aruba
  Saved to maps/countries/Non-Pacific/Aruba.jpg
  Processing: Bahamas
  Saved to maps/countries/Non-Pacific/Bahamas.jpg
  Processing: Barbados
  Saved to maps/countries/Non-Pacific/Barbados.jpg
  Processing: Belize
  Saved to maps/countries/Non-Pacific/Belize.jpg
  Processing: Bermuda
  Saved to maps/countries/Non-Pacific/Bermuda.jpg
  Processing: British Virgin Islands
  Saved to maps/countries/Non-Pacific/British_Virgin_Islands.jpg
  Processing: Cabo Verde
  Saved to maps/countries/Non-Pacific/Cabo_Verde.jpg
  Processing: Cayman Islands
  Saved to maps/countries/Non-Pacific/Cayman_Islands.jpg
  Processing: Comoros
  Saved to maps/countries/Non-Pacific/Comoros.jpg
  Processing: Cook